# 03. Train Final Models

Thin, cell-by-cell walkthrough around `scripts/train.py` -- it imports and
calls the same functions the CLI script uses (`train_final_models`,
`save_models`, `build_submission`) instead of duplicating the training
logic here. Run `python scripts/train.py` directly for the same result
with no notebook involved.

Loads config (`liverrisk/best_config.json`, written by `02_grid_search`)
and processed features (`liverrisk/data/processed/`, written by
`01_data_exploration`) -- does not regenerate either.

In [1]:
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "liverrisk" / "features.py").exists():
            return p
    raise RuntimeError("Could not locate repo root (liverrisk/features.py not found)")


REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT:", REPO_ROOT)

REPO_ROOT: c:\Users\paabl\OneDrive\Documents\GitHub\TFG_pabloCalderon


In [ ]:
import sys
if str(REPO_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "scripts"))

from liverrisk import config
from train import build_submission, save_models, train_final_models

PROCESSED_DIR = REPO_ROOT / "liverrisk" / "data" / "processed"
MODELS_DIR = REPO_ROOT / "models"
OUTPUT_PATH = REPO_ROOT / "outputs" / "improved_submission.csv"

print("Blend weights (w_cox, w_rsf, w_xgb):")
print("  hepatic:", config.blend_weights_hep())
print("  death  :", config.blend_weights_death())

## Fit final models

Coxnet (alpha-tuned), RSF (500 trees), XGB -- same procedure as `main()` in the original notebook, using the tuned per-endpoint blend weights from config.

In [3]:
result = train_final_models()

print("\nTest-set predictions (first 5 rows):")
print("risk_hepatic_event:", result["pred_hep"][:5])
print("risk_death        :", result["pred_death"][:5])

Loaded processed features: hepatic n=1253, death n=984, test n=423
Blend weights (w_cox, w_rsf, w_xgb) -- hepatic=[0.0, 1.0, 0.0], death=[0.4, 0.4, 0.2]
Fitting Coxnet (alpha CV)...


c:\Users\paabl\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
12 fits failed out of a total of 90.
The score on these train-test partitions for these parameters will be set to 0.5.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
12 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\paabl\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\paabl\AppData\Local\Programs\Python\Python312\Lib\site-packages\sksurv\linear_model\coxnet.py", line 285, in fit
    coef, alphas, deviance_ratio, n_iter = call_fit_coxnet(
                                           ^^^^^^^^^^^^^^^^


Fitting RSF (500 trees)...
Fitting XGB (survival:cox)...
Fitted 6 models.

Test-set predictions (first 5 rows):
risk_hepatic_event: [0.29314421 0.17966903 0.04491726 0.80851064 0.52245863]
risk_death        : [0.23498818 0.41371158 0.40898345 0.97115839 0.54562648]


## Save fitted models

Writes `cox_hep.joblib`, `cox_death.joblib`, `rsf_hep.joblib`, `rsf_death.joblib`, `xgb_hep.joblib`, `xgb_death.joblib`, the blend weights used, the training cohort's blended risk scores, and a copy of `feature_columns.json` -- all to `models/` (git-ignored).

In [4]:
save_models(result, models_dir=MODELS_DIR)
print("Saved models to:", MODELS_DIR)
for f in sorted(MODELS_DIR.glob("*")):
    print(" -", f.name)

Saved models to: c:\Users\paabl\OneDrive\Documents\GitHub\TFG_pabloCalderon\models
 - blend_weights_death.joblib
 - blend_weights_hep.joblib
 - cox_death.joblib
 - cox_hep.joblib
 - feature_columns.json
 - rsf_death.joblib
 - rsf_hep.joblib
 - train_scores_death.joblib
 - train_scores_hepatic.joblib
 - xgb_death.joblib
 - xgb_hep.joblib


## Build the submission CSV

In [ ]:
submission = build_submission(result, processed_dir=PROCESSED_DIR, output_path=OUTPUT_PATH)

print(f"Saved {len(submission)} predictions -> {OUTPUT_PATH}")
submission.head()